In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Observation names are not unique. To make them unique, call `.obs_names_make_unique`."
)
warnings.filterwarnings(
    "ignore",
    message= "RuntimeWarning: invalid value encountered in divide"
)


In [4]:
import scanpy as sc

In [5]:

data_origin = "Sciplex3_llm_test1_42_cellflow_True_X_pca_100_None"
results_save_path = f"results/cellflow_{data_origin}"

control_key = "is_control"
condition_keys = "perturbation"
condition_combined_keys = "condition_combined"
condition_rep_keys = "perturbation_embeddings"
mass_deduct_keys = "plate_well"
sample_rep = "X_pca"

In [ ]:

data_origin = "PBMC_all_hvg2000_gemini_test1_42_cellflow_8_True_X_pca_100_None"
results_save_path = f"results/cellflow_{data_origin}"

control_key = "is_control"
condition_keys = "cytokine"
condition_combined_keys = "condition_combined"
condition_rep_keys = "perturbation_embeddings"
mass_deduct_keys = "bc1_well"
sample_rep = "X_pca"

In [7]:
# load data
preprocess_save_path = f"./data/processed/{data_origin}"
adata_control = sc.read_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train = sc.read_h5ad(f"{preprocess_save_path}_train.h5ad")
if os.path.exists(f"{preprocess_save_path}_test.h5ad"):
    adata_test = sc.read_h5ad(f"{preprocess_save_path}_test.h5ad")
else:
    adata_test = None
print(adata_control)
print(adata_train)
print(adata_test)

AnnData object with n_obs × n_vars = 17578 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control', 'plate_well', 'cell_line_idx', 'dose_value_scaled', 'time_scaled', 'condition_combined'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cov_config', 'global_rulebook', 'hvg', 'log1p', 'normalized_m', 'pca', 'perturbation_embeddings'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'
AnnData object with n_obs × n_vars = 718049 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type', 'is_control', 'plate_well', 'cel

In [8]:
import pickle
if True:
    with open(f"{results_save_path}/inference_results.pkl", "rb") as f:
        inference_results = pickle.load(f)
        print(f"load inference_results")
    results_embedding = inference_results["results_embedding"]
    results_genes = inference_results["results_genes"]
    sampled_indices = inference_results["sampled_indices"]

load inference_results


In [12]:
from src.evaluate.util import evaluate_generated_results
final_df, eval_artifacts = evaluate_generated_results(
    results_embedding=results_embedding,
    results_genes=results_genes,
    adata_control=adata_control,
    adata_test=adata_test,
    condition_combined_keys=condition_combined_keys,
    control_key=control_key,
    sample_rep=sample_rep,
    results_save_path=results_save_path,
    mass_deduct_keys=mass_deduct_keys,
    random_seed = 42,
    Edistance_sample_num = 10000,
    dist_max_cells= 10000,
    dist_top_n_degs = 50,
    deg_padj_cutoff = 0.01,
    deg_fc_cutoff = 1.0,
    deg_top_n_for_eval = 50,
    save_each = True,
    save_final = True,
    final_filename= "final_metrics_test.csv",
    use_groupwise_control = True,
    cell_type_key = "cell_line",
    celltype_transition_rules = None,
    celltype_transition_unknown_policy="same_only",
    detailed_perturbation = False,
    detailed_celltype = False,
)


Evaluating Latent metrics (MSE, R2, PCC Delta, Wasserstein/Sinkhorn, etc.)...
Start evaluating 124 perturbations (Latent Space)
Evaluating Population Average metrics...
Start evaluating 124 perturbations (Pseudo-bulk)


/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012

Evaluating Population Distribution metrics...
Start evaluating 124 perturbations (DEG overlap, mass-aware pred)


/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012


--- Starting Stratified Evaluation ---
Training KNN (k=15) for label transfer using 'X_pca'...
Label transfer completed.
Start evaluating Cell-type specific metrics across 124 perturbations...


/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012

All evaluations completed. Results saved to results/cellflow_Sciplex3_llm_test1_42_cellflow_True_X_pca_100_None/final_metrics_test.csv


In [9]:
import yaml

with open("data/processed/PBMC_celltype_transition_rules.yaml", "r") as f:
    celltype_transition_rules = yaml.safe_load(f)


In [ ]:
from src.evaluate.util import evaluate_generated_results
final_df, eval_artifacts = evaluate_generated_results(
    results_embedding=results_embedding,
    results_genes=results_genes,
    adata_control=adata_control,
    adata_test=adata_test,
    condition_combined_keys=condition_combined_keys,
    control_key=control_key,
    sample_rep=sample_rep,
    results_save_path=results_save_path,
    mass_deduct_keys=mass_deduct_keys,
    random_seed = 42,
    Edistance_sample_num = 10000,
    dist_max_cells= 10000,
    dist_top_n_degs = 50,
    deg_padj_cutoff = 0.01,
    deg_fc_cutoff = 1.0,
    deg_top_n_for_eval = 50,
    save_each = True,
    save_final = True,
    final_filename= "final_metrics_test.csv",
    use_groupwise_control = True,
    cell_type_key = "cell_type",
    celltype_transition_rules=celltype_transition_rules,
    celltype_transition_unknown_policy="same_only",
    detailed_perturbation = False,
    detailed_celltype = False,
)


Evaluating Latent metrics (MSE, R2, PCC Delta, Wasserstein/Sinkhorn, etc.)...
Start evaluating 48 perturbations (Latent Space)
Evaluating Population Average metrics...
Start evaluating 48 perturbations (Pseudo-bulk)


/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012

Evaluating Population Distribution metrics...
Start evaluating 48 perturbations (DEG overlap, mass-aware pred)


/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012


--- Starting Stratified Evaluation ---
Training KNN (k=15) for label transfer using 'X_pca'...
Label transfer completed.
Start evaluating Cell-type specific metrics across 48 perturbations...


/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012136/software/miniconda3/envs/dvc-cellflow/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:347: RuntimeWarning: invalid value encountered in divide
  scores = (
/lustre/home/2300012

In [ ]:
print(final_df)